In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

from kbo_occultation import config
from kbo_occultation.observations.log import load_observation_log, nights_table, observations_table, validate
from kbo_occultation.observations.cc_reports import load_conditions, pointing_conditions, conditions_table
from kbo_occultation.observations.dc_combine import load_dc_series, combine_lightcurve_with_dc
from kbo_occultation.simulation.detectability import bin_average
from kbo_occultation.analysis.snr import (N_BINS, DETREND_WINDOW_S, DC_CUT_HZ, load_lightcurves, nsb_table,
                                          measured_snr_table, theoretical_snr_table, plot_snr_vs_magnitude)
from kbo_occultation.analysis.snr_measurement import (good_time_lightcurve, detrended_snr, band_noise_rms,
                                                      noise_vs_bin_size, frame_noise, variance_estimator_floor)

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.precision", 3)

log = load_observation_log()
NIGHTS = log.select()   # skips nights/observations with `exclude:` in the log; log.select(include_excluded=True) for all
NIGHT_COLORS = dict(zip(log.nights, ["tab:blue", "tab:red", "tab:green"]))
TELESCOPES = {ch: config.ORM_ARRAY.by_channel(ch) for ch in "ABC"}
MARKERS = {"MAGIC-1": "s", "MAGIC-2": "o", "LST-1": "^"}

In [2]:
import re
import yaml
import astropy.units as u
from astropy.io import fits

from kbo_occultation import PACKAGE_DATA
from kbo_occultation.simulation.instruments import Instrument, FRAME_SAMPLES_DURATION_S, reference_files_path
from kbo_occultation.analysis.snr import snr_report
from kbo_occultation.analysis.snr_measurement import throughput_from_faulty_panels
from kbo_occultation.observations.cc_reports import conditions_table

SITE = "LaPalma"
SHOW_MST = False                     # True adds the CTAO-North MST (NectarCAM) to every plot
TELS = ["MAGIC", "LST"] + (["MST-N"] if SHOW_MST else [])
INSTR = {t: Instrument(t, site=SITE) for t in TELS}
INSTR_SII = {t: Instrument(t, filter_type="MAGIC_SII", site=SITE) for t in TELS}
TEL_COLORS = {"MAGIC": "#2a78d6", "LST": "#eb6834", "MST-N": "#1baf7a"}
STEP_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
KIND = {"MAGIC-1": "MAGIC", "MAGIC-2": "MAGIC", "LST-1": "LST"}      # data telescope -> Instrument type
DC_KEY = {t.name: t.dc_key for t in config.ORM_ARRAY.telescopes}

WL = INSTR["MAGIC"].default_wavelengths.astype(float)   # 100-999 nm, 1 nm: the grid the package integrates on
BIN_MS = np.logspace(-3, 3, 61)                           # bin times for the "vs time" plots (1 us - 1 s)
MAG_GRID = np.linspace(5.5, 9.0, 36)                      # our stars are B = 5.95-8.62
REF_MAG, REF_BIN = 7.0, 10 * u.ms                         # reference star and bin for single-number comparisons
BAND = "B"                                                # magnitude band fed to the theory (as kbo-night snr)
trapz = getattr(np, "trapezoid", None) or np.trapz

def tel_axes(ncols=None, **kw):
    fig, axes = plt.subplots(1, len(TELS) if ncols is None else ncols, figsize=(6.5 * len(TELS), 5.8),
                             squeeze=False, **kw)
    return fig, axes[0]

In [3]:
TELS

['MAGIC', 'LST']